# 02 — Statistical NLP: TF-IDF, Naive Bayes, logistic regression, SVM, thresholding

This notebook is part of the NLP implementation pack for AI PMs, founders, and builders. It is designed to be runnable on toy data and easy to adapt to real company data.

## Mental model

Statistical NLP turns text into features and learns a classifier.

Classic pipeline:

```text
text → features → classifier → label
```

This is still valuable for company projects because TF-IDF baselines are cheap, fast, interpretable, and surprisingly strong for classification.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_DIR / "sample_customer_tickets.csv")
train_df, test_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["sentiment"])
train_df.head()

## TF-IDF + logistic regression

`TfidfVectorizer` converts raw documents into sparse numeric features. Logistic regression learns class weights over those features.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1, lowercase=True)),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

clf.fit(train_df["text"], train_df["sentiment"])
preds = clf.predict(test_df["text"])
print(classification_report(test_df["sentiment"], preds, zero_division=0))
print(confusion_matrix(test_df["sentiment"], preds, labels=clf.classes_))
print("Classes:", clf.classes_)

## Compare models quickly

For a portfolio project, always include a simple baseline comparison. Do not jump straight to large models.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

models = {
    "naive_bayes": MultinomialNB(),
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "linear_svm": LinearSVC(class_weight="balanced"),
}

results = []
for name, model in models.items():
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
        ("model", model),
    ])
    pipe.fit(train_df["text"], train_df["sentiment"])
    p = pipe.predict(test_df["text"])
    results.append({
        "model": name,
        "accuracy": accuracy_score(test_df["sentiment"], p),
        "macro_f1": f1_score(test_df["sentiment"], p, average="macro"),
    })

pd.DataFrame(results).sort_values("macro_f1", ascending=False)

## Inspect important features

This is a major advantage of classical ML: you can explain which words are pushing predictions.

In [ ]:
import numpy as np

best = clf
vectorizer = best.named_steps["tfidf"]
model = best.named_steps["model"]
feature_names = np.array(vectorizer.get_feature_names_out())

for class_idx, class_name in enumerate(model.classes_):
    coefs = model.coef_[class_idx]
    top = np.argsort(coefs)[-10:][::-1]
    print("\nClass:", class_name)
    print(feature_names[top].tolist())

## Error analysis

A founder-grade ML notebook always includes error analysis. This tells you whether you need better labels, rules, more data, or a stronger model.

In [ ]:
err = test_df.copy()
err["pred"] = preds
err["correct"] = err["sentiment"] == err["pred"]
err[["text", "sentiment", "pred", "correct"]]

## Threshold tuning for business trade-offs

For binary/high-risk workflows, do not blindly use the default threshold. Tune for precision or recall depending on business risk.

Example: negative complaint escalation. Missing a serious complaint may be worse than reviewing extra false positives, so you may optimize for high recall.

In [ ]:
# Binary model: negative vs not_negative
bin_train = train_df.assign(label=(train_df["sentiment"] == "negative").astype(int))
bin_test = test_df.assign(label=(test_df["sentiment"] == "negative").astype(int))

bin_clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
bin_clf.fit(bin_train["text"], bin_train["label"])
probs = bin_clf.predict_proba(bin_test["text"])[:, 1]

from sklearn.metrics import precision_score, recall_score, f1_score
rows = []
for threshold in [0.2, 0.3, 0.5, 0.7, 0.8]:
    pred = (probs >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "precision": precision_score(bin_test["label"], pred, zero_division=0),
        "recall": recall_score(bin_test["label"], pred, zero_division=0),
        "f1": f1_score(bin_test["label"], pred, zero_division=0),
    })
pd.DataFrame(rows)

## Save and load a production baseline

For a real project, add versioning, model cards, data hashes, and monitoring.

In [ ]:
import joblib
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(clf, MODEL_DIR / "tfidf_logreg_sentiment.joblib")
loaded = joblib.load(MODEL_DIR / "tfidf_logreg_sentiment.joblib")
loaded.predict(["The app is slow and keeps crashing"])[0]